## Day-4-Pandas-2

### mt1.py

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

np.random.seed(42)

sales_df = pd.DataFrame({
    "Date": pd.date_range("2025-01-01", periods=50, freq="D"),
    "Salesperson": np.random.choice(
        ["Amit", "Riya", "Karan", "Neha", "Priya"],
        50
    ),
    "Region": np.random.choice(
        ["North", "South", "East", "West"],
        50
    ),
    "Product": np.random.choice(
        ["Laptop", "Mobile", "Tablet", "Monitor"],
        50
    ),
    "Units": np.random.randint(1, 20, 50),
    "UnitPrice": np.random.randint(500, 5000, 50)
})

sales_df["Revenue"] = sales_df["Units"] * sales_df["UnitPrice"]

summary = sales_df.groupby("Region").agg(
    total_revenue=("Revenue", "sum"),
    units_sold=("Units", "sum"),
    average_deal_size=("Revenue", "mean")
).reset_index()

pivot = pd.pivot_table(
    sales_df,
    values="Revenue",
    index="Region",
    columns="Product",
    aggfunc="sum",
    fill_value=0
)

salesperson_rank = (
    sales_df.groupby("Salesperson")["Revenue"]
    .sum()
    .reset_index()
)

salesperson_rank["Rank"] = salesperson_rank["Revenue"].rank(
    method="dense",
    ascending=False
)

salesperson_rank = salesperson_rank.sort_values("Rank")

sales_df.to_csv(OUTPUT_DIR / "sales_clean_day4.csv", index=False)

with pd.ExcelWriter(OUTPUT_DIR / "sales_report_day4.xlsx") as writer:
    sales_df.to_excel(writer, sheet_name="raw", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)
    pivot.to_excel(writer, sheet_name="pivot")

print("Regional Summary:")
print(summary)

print("\nRevenue by Region and Product:")
print(pivot)

print("\nSalesperson Ranking:")
print(salesperson_rank)

print("\nFiles exported:")
print("outputs/sales_clean_day4.csv")
print("outputs/sales_report_day4.xlsx")

### mt2.py

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = sns.load_dataset("titanic")

df["age"] = df["age"].fillna(df["age"].median())
df["fare"] = df["fare"].fillna(df["fare"].median())

df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])

df["deck"] = df["deck"].cat.add_categories(["Unknown"])
df["deck"] = df["deck"].fillna("Unknown")

df["FamilySize"] = df["sibsp"] + df["parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df["AgeGroup"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

df["FareBand"] = pd.qcut(
    df["fare"],
    q=4,
    labels=["Low", "Medium", "High", "VeryHigh"],
    duplicates="drop"
)

survival_table = pd.crosstab(
    [df["AgeGroup"], df["sex"]],
    df["survived"],
    normalize="index"
) * 100

print("Survival by AgeGroup and Sex")
print(survival_table)

class_stats = df.groupby("pclass").agg(
    mean_age=("age", "mean"),
    mean_fare=("fare", "mean"),
    survival_rate=("survived", "mean"),
    passenger_count=("survived", "count")
).reset_index()

class_stats["survival_rate"] *= 100

print("\nClass Statistics")
print(class_stats)

class_info = pd.DataFrame({
    "pclass": [1, 2, 3],
    "label": ["First", "Second", "Third"],
    "deck_level": ["Upper", "Middle", "Lower"]
})

df = pd.merge(df, class_info, on="pclass", how="left")

df.to_csv(OUTPUT_DIR / "titanic_final_day4.csv", index=False)

with pd.ExcelWriter(OUTPUT_DIR / "titanic_report_day4.xlsx") as writer:
    df.to_excel(writer, sheet_name="Data", index=False)
    survival_table.to_excel(writer, sheet_name="SurvivalPivot")
    class_stats.to_excel(writer, sheet_name="ClassStats", index=False)

print("\nFiles exported:")
print("outputs/titanic_final_day4.csv")
print("outputs/titanic_report_day4.xlsx")

### mt3.py

In [ ]:
import pandas as pd

df = pd.read_csv("data.csv")

df["CustomerName"] = df["CustomerName"].str.strip().str.title()
df["Description"] = df["Description"].str.strip().str.lower()

df["Tokens"] = df["Description"].str.split()

df["HasDiscount"] = df["Description"].str.contains(
    "discount|offer|sale",
    case=False,
    na=False,
    regex=True
)

df["ProductCode"] = df["Description"].str.extract(r"([a-z]{2}\d{3})", expand=False).str.upper()

words = (
    df["Description"]
    .str.split()
    .explode()
    .value_counts()
)

print("Cleaned Data:")
print(df)

print("\nWord Frequency:")
print(words)

### mt4.py

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

np.random.seed(42)

n = 10000

df = pd.DataFrame({
    "round": range(1, n + 1),
    "die1": np.random.randint(1, 7, n),
    "die2": np.random.randint(1, 7, n)
})

df["total"] = df["die1"] + df["die2"]

stats = df.groupby("total").size().reset_index(name="frequency")
stats["probability"] = stats["frequency"] / n

prob_gt_9 = (df["total"] > 9).mean()

combo_freq = pd.pivot_table(
    df,
    index="die1",
    columns="die2",
    values="round",
    aggfunc="count",
    fill_value=0
)

print("Frequency and Probability by Total:")
print(stats)

print("\nProbability of Total > 9:")
print(prob_gt_9)

print("\nDie1 vs Die2 Frequency Table:")
print(combo_freq)

with pd.ExcelWriter(OUTPUT_DIR / "dice_simulation.xlsx") as writer:
    df.to_excel(writer, sheet_name="RawData", index=False)
    stats.to_excel(writer, sheet_name="Stats", index=False)
    combo_freq.to_excel(writer, sheet_name="Pivot")

print("\nFile exported: outputs/dice_simulation.xlsx")

### t1.py

In [ ]:
import seaborn as sns

titanic = sns.load_dataset("titanic")

top_10 = titanic.sort_values(by="fare", ascending=False)[["who", "fare", "class"]].head(10)

print(top_10)

### t2.py

In [ ]:
import seaborn as sns

titanic = sns.load_dataset("titanic")

youngest = titanic.loc[
    titanic.dropna(subset=["age"]).groupby("pclass")["age"].idxmin(),
    ["pclass", "age", "class"]
]

print(youngest)

### t3.py

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Student": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"],
    "Marks": [95, 88, 95, 76, 88, 92, 76, 85, 92, 80]
})

df["Rank_average"] = df["Marks"].rank(method="average", ascending=False)
df["Rank_min"] = df["Marks"].rank(method="min", ascending=False)
df["Rank_max"] = df["Marks"].rank(method="max", ascending=False)
df["Rank_first"] = df["Marks"].rank(method="first", ascending=False)
df["Rank_dense"] = df["Marks"].rank(method="dense", ascending=False)

print(df.sort_values("Marks", ascending=False))

### t4.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

sex_survival = df.groupby("sex")["survived"].mean() * 100
print("Survival Rate by Sex:")
print(sex_survival)

class_survival = df.groupby("pclass")["survived"].mean() * 100
print("\nSurvival Rate by Class:")
print(class_survival)

group_survival = df.groupby(["sex", "pclass"])["survived"].mean() * 100
print("\nSurvival Rate by Sex and Class:")
print(group_survival)

best_group = group_survival.idxmax()
best_rate = group_survival.max()

print("\nHighest Survival Group:")
print("Group:", best_group)
print("Rate:", best_rate)

### t5.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

fare_stats = df.groupby("pclass")["fare"].agg(["mean", "median", "std"])

print(fare_stats)

### t6.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

df["class_median_age"] = df.groupby("pclass")["age"].transform("median")

df["IsOlderThanClassMedian"] = df["age"] > df["class_median_age"]

print(df[["pclass", "age", "class_median_age", "IsOlderThanClassMedian"]].head())

### t7.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

filtered_df = df.groupby("pclass").filter(lambda x: len(x) >= 200)

print(filtered_df["pclass"].value_counts())

print("Passengers remaining:", len(filtered_df))

### t8.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

port_stats = df.groupby("embarked").agg(
    total_passengers=("survived", "count"),
    survival_rate=("survived", "mean"),
    average_fare=("fare", "mean")
)

port_stats = port_stats.sort_values("survival_rate", ascending=False)

print(port_stats)

### t9.py

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["Aryan", "Mihir", "Vidhan", "Harsh", "Dwip"],
    "age": [18, 19, 20, 18, 21]
})

grades = pd.DataFrame({
    "id": [1, 2, 2, 3, 6],
    "subject": ["Math", "Math", "Science", "Math", "Math"],
    "marks": [85, 90, 88, 75, 95]
})

inner_join = pd.merge(students, grades, on="id", how="inner")
left_join = pd.merge(students, grades, on="id", how="left")
right_join = pd.merge(students, grades, on="id", how="right")
outer_join = pd.merge(students, grades, on="id", how="outer")

print("Inner Join")
print(inner_join)

print("\nLeft Join")
print(left_join)

print("\nRight Join")
print(right_join)

print("\nOuter Join")
print(outer_join)

### t10.py

In [ ]:
import pandas as pd

jan_sales = pd.DataFrame({
    "Product": ["A", "B", "C"],
    "Sales": [1200, 1500, 1800]
})

feb_sales = pd.DataFrame({
    "Product": ["A", "B", "C"],
    "Sales": [1300, 1400, 1900]
})

jan_sales["Month"] = "January"
feb_sales["Month"] = "February"

yearly_sales = pd.concat([jan_sales, feb_sales], ignore_index=True)

print(yearly_sales)

### t11.py

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

class_labels = pd.DataFrame({
    "pclass": [1, 2, 3],
    "class_label": ["First", "Second", "Third"]
})

merged_df = pd.merge(df, class_labels, on="pclass", how="left")

print(merged_df[["pclass", "class_label"]].head())

print("\nAll rows have a label:",
      merged_df["class_label"].notna().all())

### t12.py

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "name": ["Aryan", "Mihir", "Vidhan", "Harsh"]
})

grades = pd.DataFrame({
    "id": [1, 2, 2, 3],
    "marks": [85, 90, 90, 75]
})

merged_df = pd.merge(students, grades, on="id", how="inner")

duplicates = merged_df[merged_df.duplicated()]

print("Duplicate Rows:")
print(duplicates)

clean_df = merged_df.drop_duplicates()

print("\nAfter Removing Duplicates:")
print(clean_df)

### t13.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

pt = df.pivot_table(
    values="age",
    index="pclass",
    columns="survived",
    aggfunc=["mean", "std"],
    margins=True
)

print(pt)

### t14.py

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

survival_pct = pd.crosstab(
    [df["sex"], df["embarked"]],
    df["survived"],
    normalize="index"
) * 100

print(survival_pct)

highest_group = survival_pct[1].idxmax()
highest_rate = survival_pct[1].max()

print("\nHighest Survival Group:")
print("Group:", highest_group)
print("Survival Rate:", highest_rate)

### t15.py

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

fare_table = pd.pivot_table(
    df,
    values="fare",
    index="pclass",
    columns="embarked",
    aggfunc="sum",
    fill_value=0
)

print(fare_table)

top_port = fare_table.idxmax(axis=1)

print("\nPort with Highest Revenue in Each Class:")
print(top_port)

### t16.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

port_names = {
    "C": "Cherbourg",
    "Q": "Queenstown",
    "S": "Southampton"
}

df["embarked"] = df["embarked"].map(port_names)

print(df[["embarked"]].head(10))

### t17.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

def get_outcome(row):
    if row["survived"] == 1 and row["fare"] > 50:
        return "Survived-Rich"
    elif row["survived"] == 1 and row["fare"] <= 50:
        return "Survived-Poor"
    elif row["survived"] == 0 and row["fare"] > 50:
        return "Died-Rich"
    else:
        return "Died-Poor"

df["Outcome"] = df.apply(get_outcome, axis=1)

print(df[["survived", "fare", "Outcome"]].head())

### t18.py

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

df["age_cut"] = pd.cut(df["age"], bins=5)

df["age_qcut"] = pd.qcut(df["age"], q=5)

print("Distribution using cut:")
print(df["age_cut"].value_counts().sort_index())

print("\nDistribution using qcut:")
print(df["age_qcut"].value_counts().sort_index())

### t19.py

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "A": [1.23456, 2.34567, 3.45678],
    "B": [4.56789, 5.67891, 6.78912]
})

df = df.round(2)

print(df)

### t20.py

In [ ]:
import pandas as pd

df = pd.read_csv("student.csv")

df["name"] = df["name"].str.strip().str.title()

df["first_name"] = df["name"].str.split().str[0]
df["last_name"] = df["name"].str.split().str[-1]

print(df)

### t21.py

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "product_code": ["AB-1234-XY", "CD-5678-ZZ", "EF-9012-AA"]
})

df["category"] = df["product_code"].str.extract(r"^([A-Z]{2})")
df["ID"] = df["product_code"].str.extract(r"-(\d{4})-")
df["suffix"] = df["product_code"].str.extract(r"([A-Z]{2})$")

print(df)

### t22.py

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")

print(df["who"].value_counts()[["man", "woman"]])

### t23.py

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "email": [
        "mihir@gmail.com",
        "akash@yahoo.com",
        "pratham@outlook.com"
    ]
})

df["domain"] = df["email"].str.extract(r'@([^.]+)')

print(df)

### t24.py

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = sns.load_dataset("titanic")

df = df.drop_duplicates()

df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["fare"] = df["fare"].fillna(df["fare"].median())

df["FamilySize"] = df["sibsp"] + df["parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df["AgeGroup"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

df["FareGroup"] = pd.qcut(df["fare"], 4, labels=["Low", "Medium", "High", "VeryHigh"])

stats = df.groupby(["sex", "pclass"]).agg(
    passengers=("survived", "count"),
    survival_rate=("survived", "mean"),
    avg_age=("age", "mean"),
    avg_fare=("fare", "mean")
).reset_index()

stats["survival_rate"] = stats["survival_rate"] * 100

df.to_csv(OUTPUT_DIR / "titanic_clean_day4.csv", index=False)

with pd.ExcelWriter(OUTPUT_DIR / "titanic_report_final_day4.xlsx") as writer:
    df.to_excel(writer, sheet_name="Data", index=False)
    stats.to_excel(writer, sheet_name="Stats", index=False)

print("Clean CSV exported: outputs/titanic_clean_day4.csv")
print("Excel report exported: outputs/titanic_report_final_day4.xlsx")
print("\nGroupby Statistics:")
print(stats)

### t25.py

In [ ]:
from pathlib import Path

import seaborn as sns

df = sns.load_dataset("titanic")

OUTPUT_DIR = Path("outputs") / "titanic_by_class"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for pclass in sorted(df["pclass"].dropna().unique()):
    class_df = df[df["pclass"] == pclass]
    class_df.to_csv(OUTPUT_DIR / f"titanic_class_{pclass}.csv", index=False)

print("CSV files saved successfully in outputs/titanic_by_class/.")
